# Phasse 1 Chunk 3

## `Auto grad` Deep Dive with Real data

*`Make_Regression` dataset from `sklearn.datasets`*

1. **Dataset-Centric Introduction: ``make_regression``**

**Problem Context**: Imagine you're a real estate analyst trying to predict house prices. To start, you build the simplest possible model: predicting a house's price (y) based on a single feature, like its square footage (X). Your model is a simple line: predicted_price = weight * square_footage + bias. How do you find the best weight (slope) and bias (y-intercept) that fit your data?

**Why it Matters**: This simple linear regression problem is the perfect way to visualize and understand gradients. The gradient of our loss function tells us the "slope of the error." It tells us exactly how to adjust our line's weight and bias to make it fit the data better.

* Remember The chain rule to find the derivative of the loss with respect to every single weight in the network: `∂L/∂w = (∂L/∂a) * (∂a/∂z) * (∂z/∂w)`. 
For a deep network, this is a sea of calculations.

PyTorch's **`autograd engine`** automates this entire process.

### Here is what you do

* 1. You tell PyTorch which tensors it needs to track for gradient calculations (our parameters, w and b). We do this by setting requires_grad=True.

* 2. You perform your forward pass as usual, calculating the final loss (which must be a single scalar value). PyTorch secretly builds a computation graph in the background, remembering every operation.

* 3. You call one magic function: .backward() on the loss tensor.
PyTorch traverses the graph backward, using the chain rule to automatically calculate the gradients for every tensor that had requires_grad=True.

The calculated gradients are stored in the .grad attribute of each parameter tensor.

In [1]:
import torch
from sklearn.datasets import make_regression

torch.manual_seed(42)


# 1. Loading the dataset and define parameters

# 100 houses and 1 feature i:e sq.footage

X,y = make_regression(n_samples=100,
                      n_features=1,
                      noise=10,
                      random_state=42)

# 2. Convert to tensors


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# NOTE : y needs to be reshaped to [100, 1] to match the prediction shape
X_tensor = torch.tensor(X, dtype=torch.float32, device = device)
y_tensor = torch.tensor(y, dtype=torch.float32, device=device).view(100,1)


# 3. The parameterrs we want the model to learn are w and b.
# So we must tell Pytorch that we want gradients for them.
w = torch.randn(1,1, dtype=torch.float32, device=device, requires_grad=True) # A random weight
b = torch.randn(1, dtype=torch.float32, device=device, requires_grad=True)   # A random bias


print(f"Initial w : {w.item():.4f},\nInitial b : {b.item():.4f}")


# 4. The Forward pass
# The same as before
y_pred = X_tensor @ w + b  # Matrix multiplication

# 5. Calculate loss
# We will use MSE. A standard for regression
loss1 = torch.nn.functional.mse_loss(y_pred, y_tensor)
# OR
loss2 = torch.mean((y_pred - y_tensor)**2)

print(f"Loss 1 and loss 2 are equal : {loss1 == loss2}")
print(f"Initial loss : {loss1.item():.4f}")

# 6. The Backward pass
# Pytorch traces the computations backward from `loss` to `w` to `b`
# and calculates the gradients automatically

loss1.backward()

# 7. Inspect the gradients
w_grad = w.grad
b_grad = b.grad
print("\n   After backward pass   ")
print(f"Gradient w : {w_grad.item():.4f},\nGradient b : {b_grad.item():.4f}")


print("\nWhat does w.grad mean?")
print("It means if we increase 'w' by a tiny amount," \
"\nthe loss will increase by 'w.grad' times that amount." \
"\nTo DECREASE the loss," \
"\nwe need to move in the opposite direction of the gradient")


Initial w : 0.1940,
Initial b : 0.1391
Loss 1 and loss 2 are equal : True
Initial loss : 1689.0951

   After backward pass   
Gradient w : -72.9922,
Gradient b : 7.1370

What does w.grad mean?
It means if we increase 'w' by a tiny amount,
the loss will increase by 'w.grad' times that amount.
To DECREASE the loss,
we need to move in the opposite direction of the gradient


# Explorations

## Multi-Level Explorations
**Beginner**:

 What happens if you forget to set requires_grad=True for w and b and then call loss.backward()? Try it. Read the error message carefully. It's one of the most common errors for beginners.

**Intermediate**:

 In the code, gradients are calculated and stored. Now, call loss.backward() a second time right after the first one. Print w.grad again. Did the value change? This is gradient accumulation. PyTorch adds new gradients to existing ones. Why is this the default behavior, and why must we manually reset gradients to zero in a training loop?

**Advanced**:

 The chain rule is dL/dw = dL/dy_pred * dy_pred/dw. For our model, y_pred = X*w + b, so dy_pred/dw is simply X. For MSE loss, L = (y_pred - y)^2, so dL/dy_pred is 2 * (y_pred - y). Calculate dL/dw manually for our problem (you'll need to average over the batch) and verify that your result is close to what PyTorch computed in w.grad.

## 5. Dataset-Specific Exercises

**Replication**:

 Use the California Housing dataset. Pick one feature ('AveRooms') as X and the target ('MedHouseVal') as y. Perform one forward and backward pass just like in the main example. Print the initial loss and the calculated gradients for w and b.

**Modification**:

 Use make_regression but with n_features=5. This means your feature tensor `X_tensor` will have shape [100, 5]. To handle this, your weight w must now be a tensor of shape [5, 1]. Perform the forward pass (y_pred = X_tensor @ w + b), calculate the loss, run .backward(), and inspect the shape of w.grad. Does it match the shape of w? Why is this logically necessary?

**Creation**:

 Let's prove you understand the computation graph. Create a simple graph from scratch:
```
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
c = a * b
d = torch.sin(c)
L = d * 2
```
Run `L.backward()`. What are the final values of `a.grad` and `b.grad`? Show your work tracing the chain rule by hand to confirm the result.

## 6. Professional Best Practices :

**Gradient Accumulation and zero_grad()**:

 The "Intermediate" exploration reveals a critical behavior. In every training step, before you call `.backward()`, you MUST explicitly clear the old gradients using optimizer`.zero_grad()`. Otherwise, you'll be accumulating gradients from all previous steps, which will throw your training off course.

**The Computation Graph is Dynamic**:

 PyTorch builds the graph on-the-fly as you execute the forward pass. This makes it incredibly flexible for dynamic models like RNNs and allows for easy debugging, a key advantage over older frameworks like TensorFlow 1.x.

